# NB-04 — めぐ指数: TrackSpeedIndex (TSI) 整合性検証

**目的**: `TSI_offset` が馬場の速さ（時計の出やすさ）を正確に捉えているかを JRA 公式の馬場状態データ（クッション値・含水率）と照合して検証する

**検証観点**:
1. TSI_offset とクッション値の相関（芝: 高クッション = 速い → 正相関）
2. TSI_offset と含水率の相関（芝: 高含水 = 遅い → 負相関）
3. TSI_offset と馬場状態カテゴリ（良/稍重/重/不良）との整合性
4. ダート vs 芝での TSI の挙動差異

**前提**: `jra_cushion` テーブルにクッション値・含水率が蓄積されていること

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

INPUT_NB01 = Path('output/nb01')
OUTPUT_DIR = Path('output/nb04')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_race = pd.read_parquet(INPUT_NB01 / 'megu_dataset.parquet')
print(f'レース結果: {len(df_race):,} 行')

## 1. JRA クッション値・含水率データの読み込み

In [ ]:
from src.db.session import get_session, init_engine
from sqlalchemy import text

init_engine()

with get_session() as session:
    df_cushion = pd.read_sql(text("""
        SELECT
            race_date,
            course,
            cushion_value,
            moisture_turf,
            moisture_dirt
        FROM jra_cushion
        WHERE race_date >= '2018-01-01'
    """), session.bind)

print(f'jra_cushion: {len(df_cushion):,} 行')
print(df_cushion.head())
print(f'\nクッション値の分布:')
print(df_cushion['cushion_value'].describe())

## 2. TSI_offset の確認・計算

In [ ]:
# TSI が既存実装から取得できている場合は直接使用
# ない場合はレースデータから推定する簡易版を計算

if 'tsi_offset' not in df_race.columns or df_race['tsi_offset'].isna().all():
    print('TSI_offset が存在しないため、日別ペース補正値から推定します')
    
    # 同一日×コース×距離×芝ダートでの平均タイム偏差をTSI代理変数とする
    grp = ['race_date', 'course', 'distance', 'surface']
    df_race['day_course_mean'] = df_race.groupby(grp)['finish_time_sec'].transform('mean')
    dist_grp = ['distance', 'surface', 'track_condition']
    df_race['dist_overall_mean'] = df_race.groupby(dist_grp)['finish_time_sec'].transform('mean')
    df_race['tsi_proxy'] = df_race['day_course_mean'] - df_race['dist_overall_mean']
    tsi_col = 'tsi_proxy'
    print('TSI代理変数 (tsi_proxy) を使用')
else:
    tsi_col = 'tsi_offset'
    print(f'既存の {tsi_col} を使用')

print(f'\nTSI の基本統計:')
print(df_race[tsi_col].describe())

In [ ]:
# レースデータとクッション値をマージ
df_merged = df_race.merge(
    df_cushion[['race_date', 'course', 'cushion_value', 'moisture_turf', 'moisture_dirt']],
    on=['race_date', 'course'],
    how='left'
)

print(f'マージ後: {len(df_merged):,} 行')
print(f'クッション値マッチ率: {df_merged["cushion_value"].notna().mean()*100:.1f}%')

## 3. TSI vs クッション値・含水率の相関

In [ ]:
df_turf = df_merged[(df_merged['surface'] == '芝') & df_merged['cushion_value'].notna() & df_merged[tsi_col].notna()]
df_dirt = df_merged[(df_merged['surface'] == 'ダート') & df_merged['moisture_dirt'].notna() & df_merged[tsi_col].notna()]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 芝: TSI vs クッション値
if len(df_turf) > 30:
    r_cushion, p_cushion = stats.pearsonr(df_turf['cushion_value'], df_turf[tsi_col])
    axes[0].scatter(df_turf['cushion_value'], df_turf[tsi_col], alpha=0.2, s=3)
    axes[0].set_xlabel('クッション値')
    axes[0].set_ylabel(f'{tsi_col}（秒）')
    axes[0].set_title(f'芝: TSI vs クッション値\nr={r_cushion:.3f}, p={p_cushion:.4f}')
    print(f'芝 TSI vs クッション値: r={r_cushion:.4f}, p={p_cushion:.4f}')
    print('  期待: 正相関（高クッション = 速い馬場 = TSI が低い〈時計が速い〉）')
else:
    axes[0].text(0.5, 0.5, 'データ不足', ha='center', va='center')

# 芝: TSI vs 含水率
if len(df_turf) > 30 and df_turf['moisture_turf'].notna().sum() > 30:
    sub = df_turf.dropna(subset=['moisture_turf'])
    r_mois, p_mois = stats.pearsonr(sub['moisture_turf'], sub[tsi_col])
    axes[1].scatter(sub['moisture_turf'], sub[tsi_col], alpha=0.2, s=3, color='coral')
    axes[1].set_xlabel('含水率（芝）')
    axes[1].set_ylabel(f'{tsi_col}（秒）')
    axes[1].set_title(f'芝: TSI vs 含水率\nr={r_mois:.3f}, p={p_mois:.4f}')
    print(f'芝 TSI vs 含水率: r={r_mois:.4f}, p={p_mois:.4f}')
    print('  期待: 正相関（高含水 = 重い馬場 = TSI が高い〈時計が遅い〉）')
else:
    axes[1].text(0.5, 0.5, 'データ不足', ha='center', va='center')

# 馬場状態カテゴリ別のTSI分布
tc_order = ['良', '稍重', '重', '不良']
tc_data = [df_merged[df_merged['track_condition'] == tc][tsi_col].dropna() for tc in tc_order]
tc_labels = [f'{tc}\n(n={len(d)})' for tc, d in zip(tc_order, tc_data)]
axes[2].boxplot([d for d in tc_data if len(d) > 0], labels=[l for d, l in zip(tc_data, tc_labels) if len(d) > 0])
axes[2].set_ylabel(f'{tsi_col}（秒）')
axes[2].set_title('馬場状態別 TSI 分布')
axes[2].axhline(0, color='red', linestyle='--')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'tsi_validation.png', dpi=120)
plt.show()

## 4. 馬場状態カテゴリとの整合性チェック

In [ ]:
# 「良」より「重」の方が TSI_offset が正（遅い）であるはずの単調性チェック
tc_means = df_merged.groupby(['surface', 'track_condition'])[tsi_col].agg(['mean', 'median', 'count'])
print('=== 馬場状態別 TSI の平均・中央値 ===')
print(tc_means)

print()
print('期待パターン: 良 < 稍重 < 重 < 不良 （正値 = 遅い馬場）')

for surface in ['芝', 'ダート']:
    print(f'\n{surface}:')
    vals = []
    for tc in ['良', '稍重', '重', '不良']:
        try:
            v = tc_means.loc[(surface, tc), 'mean']
            vals.append(v)
            print(f'  {tc}: {v:.3f}秒')
        except KeyError:
            vals.append(None)
            print(f'  {tc}: データなし')
    nonnone = [v for v in vals if v is not None]
    if len(nonnone) >= 2:
        is_monotone = all(nonnone[i] <= nonnone[i+1] for i in range(len(nonnone)-1))
        print(f'  → 単調増加: {"✅" if is_monotone else "⚠️"}')

## 5. TSI 計算式の改善提案（必要な場合）

In [ ]:
# TSI の精度が低い場合の代替計算式: クッション値・含水率を直接使う回帰
df_candidate = df_turf.dropna(subset=['cushion_value', 'moisture_turf', tsi_col]).copy()

if len(df_candidate) > 100:
    import statsmodels.formula.api as smf
    formula = f'{tsi_col} ~ cushion_value + moisture_turf'
    alt_model = smf.ols(formula, data=df_candidate).fit()
    print('=== 代替 TSI 計算式（クッション値 + 含水率）===')
    print(alt_model.summary().tables[1])
    print(f'R²: {alt_model.rsquared:.4f}')
    print()
    print('代替式: TSI_alt = β₀ + β₁×クッション値 + β₂×含水率（芝）')
else:
    print('代替モデル推定に必要なデータが不足しています')

# 結論
print('\n=== TSI 検証結論 ===')
print('1. TSI と馬場状態の整合性確認 → 上記の単調性チェック参照')
print('2. クッション値との相関 → 上記相関係数参照')
print('3. めぐ指数への影響: β₂ × TSI_offset の補正量の妥当性は NB-06 有効性検証で最終判断')

# 日別 TSI の分布を保存
daily_tsi = (
    df_race.groupby(['race_date', 'course', 'surface'])[tsi_col]
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
daily_tsi.to_parquet(OUTPUT_DIR / 'daily_tsi_summary.parquet', index=False)
print(f'\n日別TSIサマリーを保存: {OUTPUT_DIR / "daily_tsi_summary.parquet"}')